# AI SQL Query Builder

Convert natural language to SQL queries using AI models via OpenRouter.

**Features:**
- Generate SQL from natural language
- Explain existing SQL queries
- Optimize slow queries
- Support for multiple SQL dialects
- Schema-aware query generation
- Access to 100+ AI models through OpenRouter

## Setup and Imports

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

## Load API Key

In [ ]:
# Load from root directory
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"✓ OpenRouter API Key found (begins {openrouter_api_key[:10]})")
else:
    print("✗ OpenRouter API Key not set")
    print("Please set OPENROUTER_API_KEY in your .env file")
    print("Get your key at: https://openrouter.ai/")

## Initialize OpenRouter Client

In [ ]:
# Single client for all models via OpenRouter
client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

print("✓ OpenRouter client initialized successfully")
print("  Access to 100+ AI models through one API")

## Model Configuration

OpenRouter gives you access to many models. Choose based on your needs:

In [ ]:
# Available models (pick what you need)
MODELS = {
    # Premium models - best quality
    "gpt5": "openai/gpt-5",
    "claude": "anthropic/claude-sonnet-4.5",
    "gemini": "google/gemini-2.5-pro",
    
    # Budget models - great value
    "deepseek": "deepseek/deepseek-chat",
    "qwen": "qwen/qwen-2.5-coder-32b-instruct",
    "llama": "meta-llama/llama-3.3-70b-instruct",
    
    # Nano models - ultra cheap
    "gpt-nano": "openai/gpt-5-nano",
}

# Default model (feel free to change)
DEFAULT_MODEL = MODELS["gpt5"]

print("Available models:")
for name, model_id in MODELS.items():
    print(f"  {name}: {model_id}")
print(f"\nDefault: {DEFAULT_MODEL}")

## System Prompts

In [ ]:
def system_prompt_for_generation():
    return """
You are an expert SQL developer. Your task is to convert natural language descriptions into accurate, efficient SQL queries.

Guidelines:
- Generate clean, well-formatted SQL queries
- Use proper indentation and capitalization
- Add brief comments for complex logic
- Choose efficient query patterns
- Use appropriate indexes and JOINs
- Consider performance implications
- Return ONLY the SQL query, no explanations unless asked
"""

def system_prompt_for_explanation():
    return """
You are an expert SQL teacher. Your task is to explain SQL queries in clear, simple language.

Guidelines:
- Break down the query step by step
- Explain what each clause does
- Mention any performance considerations
- Highlight best practices or potential issues
- Use simple language that beginners can understand
"""

def system_prompt_for_optimization():
    return """
You are an expert SQL performance tuner. Your task is to analyze SQL queries and suggest optimizations.

Guidelines:
- Identify performance bottlenecks
- Suggest index improvements
- Recommend better JOIN strategies
- Propose query rewrites when beneficial
- Explain the reasoning behind each suggestion
- Provide the optimized query
"""

print("✓ System prompts defined")

## Core Functions

In [ ]:
def generate_sql(description, schema="", dialect="PostgreSQL", model=None):
    """
    Generate SQL query from natural language description.
    
    Args:
        description: Natural language description of what you want
        schema: Database schema (optional but recommended)
        dialect: SQL dialect (PostgreSQL, MySQL, SQLite, SQL Server)
        model: Model to use (default: DEFAULT_MODEL). Can be model key or full model ID
    """
    # Get model ID
    if model is None:
        model_id = DEFAULT_MODEL
    elif model in MODELS:
        model_id = MODELS[model]
    else:
        model_id = model  # Assume it's a full model ID
    
    schema_context = f"\n\nDatabase Schema:\n{schema}" if schema else ""
    
    user_prompt = f"""
Generate a {dialect} SQL query for the following request:

{description}{schema_context}

Return only the SQL query, properly formatted.
"""
    
    messages = [
        {"role": "system", "content": system_prompt_for_generation()},
        {"role": "user", "content": user_prompt}
    ]
    
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0.3
    )
    
    sql = response.choices[0].message.content
    # Clean up markdown code blocks if present
    sql = sql.replace('```sql', '').replace('```', '').strip()
    
    return sql


def explain_sql(query, model=None):
    """
    Explain what an SQL query does.
    
    Args:
        query: SQL query to explain
        model: Model to use (default: DEFAULT_MODEL)
    """
    # Get model ID
    if model is None:
        model_id = DEFAULT_MODEL
    elif model in MODELS:
        model_id = MODELS[model]
    else:
        model_id = model
    
    user_prompt = f"""
Explain this SQL query in simple terms:

```sql
{query}
```

Break it down step by step and mention any important considerations.
"""
    
    messages = [
        {"role": "system", "content": system_prompt_for_explanation()},
        {"role": "user", "content": user_prompt}
    ]
    
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0.5
    )
    
    return response.choices[0].message.content


def optimize_sql(query, context="", model=None):
    """
    Get optimization suggestions for an SQL query.
    
    Args:
        query: SQL query to optimize
        context: Additional context (table sizes, performance issues, etc.)
        model: Model to use (default: DEFAULT_MODEL)
    """
    # Get model ID
    if model is None:
        model_id = DEFAULT_MODEL
    elif model in MODELS:
        model_id = MODELS[model]
    else:
        model_id = model
    
    context_info = f"\n\nAdditional Context:\n{context}" if context else ""
    
    user_prompt = f"""
Analyze and optimize this SQL query:

```sql
{query}
```{context_info}

Provide:
1. Performance analysis
2. Optimization suggestions
3. Optimized version of the query
"""
    
    messages = [
        {"role": "system", "content": system_prompt_for_optimization()},
        {"role": "user", "content": user_prompt}
    ]
    
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0.3
    )
    
    return response.choices[0].message.content


print("✓ Core functions defined successfully")

## Example Database Schema

Let's define a sample e-commerce database schema for our examples:

In [ ]:
ECOMMERCE_SCHEMA = """
users (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100) UNIQUE,
    country VARCHAR(50),
    created_at TIMESTAMP DEFAULT NOW()
)

products (
    id SERIAL PRIMARY KEY,
    name VARCHAR(200),
    category VARCHAR(50),
    price DECIMAL(10,2),
    stock_quantity INTEGER,
    created_at TIMESTAMP DEFAULT NOW()
)

orders (
    id SERIAL PRIMARY KEY,
    user_id INTEGER REFERENCES users(id),
    total_amount DECIMAL(10,2),
    status VARCHAR(20),
    created_at TIMESTAMP DEFAULT NOW()
)

order_items (
    id SERIAL PRIMARY KEY,
    order_id INTEGER REFERENCES orders(id),
    product_id INTEGER REFERENCES products(id),
    quantity INTEGER,
    price DECIMAL(10,2)
)

reviews (
    id SERIAL PRIMARY KEY,
    user_id INTEGER REFERENCES users(id),
    product_id INTEGER REFERENCES products(id),
    rating INTEGER CHECK (rating >= 1 AND rating <= 5),
    comment TEXT,
    created_at TIMESTAMP DEFAULT NOW()
)
"""

print("✓ Example schema defined")

## Example 1: Simple Query Generation

In [ ]:
description = "Get all users who registered in the last 7 days"

print(f"Request: {description}\n")
print("Generated SQL:")
print("=" * 60)

sql = generate_sql(description, schema=ECOMMERCE_SCHEMA)
print(sql)

## Example 2: Complex Query with JOINs

In [ ]:
description = """
Find the top 10 customers by total purchase amount in the last 30 days.
Show their name, email, number of orders, and total spent.
Only include customers with at least 2 orders.
"""

print(f"Request: {description}\n")
print("Generated SQL:")
print("=" * 60)

sql = generate_sql(description, schema=ECOMMERCE_SCHEMA)
print(sql)

## Example 3: Analytics Query

In [ ]:
description = """
Calculate the average rating and total number of reviews for each product category.
Show only categories with at least 10 reviews.
Order by average rating descending.
"""

print(f"Request: {description}\n")
print("Generated SQL:")
print("=" * 60)

sql = generate_sql(description, schema=ECOMMERCE_SCHEMA)
print(sql)

## Example 4: Explain an SQL Query

In [ ]:
complex_query = """
SELECT 
    u.name,
    COUNT(DISTINCT o.id) as order_count,
    SUM(o.total_amount) as total_spent,
    AVG(o.total_amount) as avg_order_value
FROM users u
INNER JOIN orders o ON u.id = o.user_id
WHERE o.created_at >= NOW() - INTERVAL '90 days'
    AND o.status = 'completed'
GROUP BY u.id, u.name
HAVING COUNT(DISTINCT o.id) >= 3
ORDER BY total_spent DESC
LIMIT 20;
"""

print("Query to explain:")
print("=" * 60)
print(complex_query)
print("\nExplanation:")
print("=" * 60)

explanation = explain_sql(complex_query)
display(Markdown(explanation))

## Example 5: Query Optimization

In [ ]:
slow_query = """
SELECT *
FROM orders o
WHERE o.user_id IN (
    SELECT u.id
    FROM users u
    WHERE u.country = 'USA'
)
AND o.created_at > '2024-01-01';
"""

context = "The users table has 1 million rows, orders table has 5 million rows. This query is taking 30+ seconds."

print("Query to optimize:")
print("=" * 60)
print(slow_query)
print(f"\nContext: {context}\n")
print("Optimization Analysis:")
print("=" * 60)

optimization = optimize_sql(slow_query, context=context)
display(Markdown(optimization))

## Example 6: Compare Different Models

Let's see how different AI models handle the same query:

In [ ]:
description = """
Find products that are low on stock (less than 10 items) and have good reviews (average rating >= 4).
Show product name, current stock, average rating, and number of reviews.
"""

print(f"Request: {description}\n")
print("="*70)

# Compare different models
models_to_compare = ["gpt5", "claude", "deepseek"]

for model_key in models_to_compare:
    print(f"\n📌 {model_key.upper()} ({MODELS[model_key]}):")
    print("="*70)
    try:
        sql = generate_sql(description, schema=ECOMMERCE_SCHEMA, model=model_key)
        print(sql)
    except Exception as e:
        print(f"Error: {e}")

## Interactive Section: Try Your Own!

Now it's your turn. Modify the variables below to generate your own queries:

In [ ]:
# Your custom request
my_description = """
Your natural language description here...
Example: Find all products in the Electronics category with price > 500
"""

# Optional: Use your own schema or use ECOMMERCE_SCHEMA
my_schema = ECOMMERCE_SCHEMA

# Choose: "PostgreSQL", "MySQL", "SQLite", "SQL Server"
my_dialect = "PostgreSQL"

# Choose model: "gpt5", "claude", "gemini", "deepseek", "qwen", "llama"
my_model = "gpt5"

# Generate!
print("Your Custom Query:")
print("="*60)
result = generate_sql(my_description, schema=my_schema, dialect=my_dialect, model=my_model)
print(result)

## More Examples: Real-World Scenarios

In [ ]:
# Customer Lifetime Value
print("Example: Customer Lifetime Value Calculation")
print("="*60)
clv_query = generate_sql(
    "Calculate customer lifetime value: total revenue per customer, their first purchase date, last purchase date, and total number of orders",
    schema=ECOMMERCE_SCHEMA
)
print(clv_query)
print()

In [ ]:
# Cohort Analysis
print("Example: Monthly Cohort Analysis")
print("="*60)
cohort_query = generate_sql(
    "Create a cohort analysis showing user retention by their signup month. Show what percentage of users from each signup month made a purchase in subsequent months",
    schema=ECOMMERCE_SCHEMA
)
print(cohort_query)
print()

In [ ]:
# Product Recommendation
print("Example: Products Frequently Bought Together")
print("="*60)
recommendation_query = generate_sql(
    "Find products that are frequently bought together. For each product pair, show how many times they were purchased in the same order",
    schema=ECOMMERCE_SCHEMA
)
print(recommendation_query)

## Budget-Friendly Testing

Try the same query with different models to compare cost vs quality:

In [ ]:
test_query = "Get top 5 selling products this month with revenue"

print("Testing budget vs premium models:\n")

# Premium
print("💎 Premium (GPT-5):")
print("="*60)
print(generate_sql(test_query, schema=ECOMMERCE_SCHEMA, model="gpt5"))

print("\n💰 Budget (DeepSeek):")
print("="*60)
print(generate_sql(test_query, schema=ECOMMERCE_SCHEMA, model="deepseek"))

print("\n⚡ Ultra Budget (GPT Nano):")
print("="*60)
print(generate_sql(test_query, schema=ECOMMERCE_SCHEMA, model="gpt-nano"))

## Tips for Best Results

1. **Be Specific**: The more details you provide, the better the query
2. **Include Schema**: Always provide your database schema when possible
3. **Mention Performance**: Add words like "efficient" or "optimized" for better queries
4. **Specify Dialect**: Different databases have different syntax
5. **Test and Iterate**: Start simple, then add complexity
6. **Try Different Models**: Budget models work great for simple queries

## Safety Notes

- Always review generated queries before running them
- Be careful with DELETE, UPDATE, and DROP operations
- Test on a development database first
- Use transactions for data modifications
- Never run queries on production without reviewing them

## Interactive UI

Let's create a beautiful Gradio interface for easy interaction:

In [ ]:
import gradio as gr

In [ ]:
def generate_sql_ui(description, schema, dialect, model):
    """Wrapper for UI"""
    try:
        sql = generate_sql(description, schema, dialect, model)
        return sql
    except Exception as e:
        return f"Error: {str(e)}"

def explain_sql_ui(query, model):
    """Wrapper for UI"""
    try:
        explanation = explain_sql(query, model)
        return explanation
    except Exception as e:
        return f"Error: {str(e)}"

def optimize_sql_ui(query, context, model):
    """Wrapper for UI"""
    try:
        optimization = optimize_sql(query, context, model)
        return optimization
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Create Gradio interface
with gr.Blocks(title="AI SQL Query Builder", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🗄️ AI SQL Query Builder")
    gr.Markdown("Convert natural language to SQL using 100+ AI models via OpenRouter")
    
    with gr.Tabs():
        # Tab 1: Generate SQL
        with gr.Tab("✨ Generate SQL"):
            with gr.Row():
                with gr.Column(scale=2):
                    gen_description = gr.Textbox(
                        label="What do you want to query?",
                        placeholder="Example: Find all users who registered in the last 7 days",
                        lines=3
                    )
                    gen_schema = gr.Textbox(
                        label="Database Schema (optional but recommended)",
                        value=ECOMMERCE_SCHEMA,
                        lines=10
                    )
                    with gr.Row():
                        gen_dialect = gr.Dropdown(
                            choices=["PostgreSQL", "MySQL", "SQLite", "SQL Server"],
                            value="PostgreSQL",
                            label="SQL Dialect"
                        )
                        gen_model = gr.Dropdown(
                            choices=list(MODELS.keys()),
                            value="gpt5",
                            label="AI Model"
                        )
                    gen_button = gr.Button("Generate SQL", variant="primary")
                
                with gr.Column(scale=2):
                    gen_output = gr.Code(
                        label="Generated SQL",
                        language="sql",
                        lines=20
                    )
            
            gen_button.click(
                fn=generate_sql_ui,
                inputs=[gen_description, gen_schema, gen_dialect, gen_model],
                outputs=gen_output
            )
            
            # Examples
            gr.Examples(
                examples=[
                    ["Get all users who registered in the last 7 days", ECOMMERCE_SCHEMA, "PostgreSQL", "gpt5"],
                    ["Find top 10 customers by total purchase amount in the last 30 days", ECOMMERCE_SCHEMA, "PostgreSQL", "claude"],
                    ["Calculate average rating per product category with at least 10 reviews", ECOMMERCE_SCHEMA, "PostgreSQL", "gpt5"],
                ],
                inputs=[gen_description, gen_schema, gen_dialect, gen_model],
            )
        
        # Tab 2: Explain SQL
        with gr.Tab("📚 Explain SQL"):
            with gr.Row():
                with gr.Column(scale=2):
                    explain_query = gr.Code(
                        label="SQL Query to Explain",
                        language="sql",
                        lines=10
                    )
                    explain_model = gr.Dropdown(
                        choices=list(MODELS.keys()),
                        value="gpt5",
                        label="AI Model"
                    )
                    explain_button = gr.Button("Explain Query", variant="primary")
                
                with gr.Column(scale=2):
                    explain_output = gr.Markdown(label="Explanation")
            
            explain_button.click(
                fn=explain_sql_ui,
                inputs=[explain_query, explain_model],
                outputs=explain_output
            )
        
        # Tab 3: Optimize SQL
        with gr.Tab("⚡ Optimize SQL"):
            with gr.Row():
                with gr.Column(scale=2):
                    optimize_query = gr.Code(
                        label="SQL Query to Optimize",
                        language="sql",
                        lines=10
                    )
                    optimize_context = gr.Textbox(
                        label="Context (optional)",
                        placeholder="Example: The users table has 1 million rows, orders table has 5 million rows",
                        lines=3
                    )
                    optimize_model = gr.Dropdown(
                        choices=list(MODELS.keys()),
                        value="claude",
                        label="AI Model"
                    )
                    optimize_button = gr.Button("Optimize Query", variant="primary")
                
                with gr.Column(scale=2):
                    optimize_output = gr.Markdown(label="Optimization Suggestions")
            
            optimize_button.click(
                fn=optimize_sql_ui,
                inputs=[optimize_query, optimize_context, optimize_model],
                outputs=optimize_output
            )
        
        # Tab 4: Model Comparison
        with gr.Tab("🔬 Compare Models"):
            gr.Markdown("### Compare how different AI models generate SQL")
            with gr.Row():
                compare_description = gr.Textbox(
                    label="What do you want to query?",
                    placeholder="Example: Find products with low stock and good reviews",
                    lines=3
                )
            with gr.Row():
                compare_model1 = gr.Dropdown(choices=list(MODELS.keys()), value="gpt5", label="Model 1")
                compare_model2 = gr.Dropdown(choices=list(MODELS.keys()), value="claude", label="Model 2")
                compare_model3 = gr.Dropdown(choices=list(MODELS.keys()), value="deepseek", label="Model 3")
            
            compare_button = gr.Button("Compare Models", variant="primary")
            
            with gr.Row():
                compare_output1 = gr.Code(label="Model 1 Result", language="sql", lines=15)
                compare_output2 = gr.Code(label="Model 2 Result", language="sql", lines=15)
                compare_output3 = gr.Code(label="Model 3 Result", language="sql", lines=15)
            
            def compare_models(desc, m1, m2, m3):
                r1 = generate_sql_ui(desc, ECOMMERCE_SCHEMA, "PostgreSQL", m1)
                r2 = generate_sql_ui(desc, ECOMMERCE_SCHEMA, "PostgreSQL", m2)
                r3 = generate_sql_ui(desc, ECOMMERCE_SCHEMA, "PostgreSQL", m3)
                return r1, r2, r3
            
            compare_button.click(
                fn=compare_models,
                inputs=[compare_description, compare_model1, compare_model2, compare_model3],
                outputs=[compare_output1, compare_output2, compare_output3]
            )
    
    gr.Markdown("""
    ---
    ### 💡 Tips
    - Be specific in your descriptions for better results
    - Include your database schema for accurate queries
    - Try different models to see which works best for your use case
    - Premium models (GPT-5, Claude) are best for complex queries
    - Budget models (DeepSeek, Qwen) work great for simple queries
    """)

print("✓ Gradio interface created")

In [ ]:
# Launch the interface
demo.launch(share=True)

## Summary

You've learned how to:
- Generate SQL queries from natural language
- Explain complex SQL queries
- Optimize slow queries
- Compare different AI models via OpenRouter
- Handle real-world analytics scenarios
- Balance cost and quality with model selection

**Why OpenRouter?**
- One API key for 100+ models
- Transparent pricing
- Easy model switching
- Free tier available
- No vendor lock-in

**Next Steps:**
- Try with your own database schema
- Experiment with different SQL dialects
- Compare model performance for your use cases
- Build a query library for common tasks
- Explore more models at [openrouter.ai/models](https://openrouter.ai/models)